# Open-Meteo

https://open-meteo.com/en/docs

In [3]:
from pathlib import Path

import openmeteo_requests

import pandas as pd
import yaml
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

config_candidates = [
    Path("configs/open_meteo.yaml"),
    Path("../configs/open_meteo.yaml"),
]
config_path = next(path for path in config_candidates if path.exists())

with config_path.open(encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

url = config["url"]
params = {
    "latitude": config["latitude"],
    "longitude": config["longitude"],
    "hourly": ",".join(config["hourly"]),
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
    "date": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
        end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    )
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 40.1875°N -8.4375°E
Elevation: 82.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m
0   2026-10-05 00:00:00+00:00       17.077999
1   2026-10-05 01:00:00+00:00       16.477999
2   2026-10-05 02:00:00+00:00       16.028000
3   2026-10-05 03:00:00+00:00       15.378000
4   2026-10-05 04:00:00+00:00       15.078000
..                        ...             ...
163 2026-10-11 19:00:00+00:00       18.881001
164 2026-10-11 20:00:00+00:00       17.331001
165 2026-10-11 21:00:00+00:00       16.131001
166 2026-10-11 22:00:00+00:00       15.331000
167 2026-10-11 23:00:00+00:00       14.831000

[168 rows x 2 columns]
